# HoaXGY - Fake News Detection for Android Application

This project implements an *end-to-end* text classification workflow to detect fake news in Indonesian. The experiment evolves from a traditional *baseline* model (TF-IDF + Logistic Regression) to a Deep Learning architecture (Multi-Scale Conv1D) optimized using **Post-Training Quantization (TFLite)** to ensure it is ready for deployment on Android devices with high performance and efficient memory usage.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import re
import time
import random
import joblib
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix, f1_score

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Dense, Dropout, Embedding, Conv1D, GlobalMaxPool1D

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [3]:
DATA_DIR = "/content/drive/MyDrive/Projects/Fake News Detection/dataset/"
cnn_real = pd.read_excel(DATA_DIR + "cnn_cleaned.xlsx")
kompas_real = pd.read_excel(DATA_DIR + "kompas_cleaned.xlsx")
tempo_real = pd.read_excel(DATA_DIR + "tempo_cleaned.xlsx")
tbh_fake = pd.read_excel(DATA_DIR + "turnbackhoax_cleaned.xlsx")

## Data Cleaning

Raw news datasets contain biases such as journalists' initials, external reference links, location tags, etc. If not cleaned, the model will suffer from *data leakage*, it will predict the authenticity of a news article based solely on the presence or absence of these initials, rather than on the substance of the narrative.

At this stage, the following steps are performed:
* Extract the original narrative from the hoax news story.
* Remove URLs, HTML tags, mentions, hashtags, and multiple spaces.
* Remove news portal names, location tags, journalists' names and initials, and potential boilerplate text.

In [4]:
def extract_narasi_only(text):
    if not isinstance(text, str):
        return ""

    pattern = r'\[?narasi\]?\W*(.*?)(?=\n\s*={3,}|\n\s*\[|\Z|penjelasan)'
    match = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
    if match:
        extracted = match.group(1).strip()
        extracted = re.sub(r'^[“"\'”]|[“"\'”]$', '', extracted)
        return extracted.strip()

    return ""

tbh_fake['clean_text'] = tbh_fake['FullText'].apply(extract_narasi_only)

In [5]:
def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Remove mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags
    text = re.sub(r'#\w+', '', text)
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip().lower()

cnn_real['clean_text'] = cnn_real['text_new'].apply(clean_text)
kompas_real['clean_text'] = kompas_real['text_new'].apply(clean_text)
tempo_real['clean_text'] = tempo_real['text_new'].apply(clean_text)
tbh_fake['clean_text'] = tbh_fake['clean_text'].apply(clean_text)

In [6]:
def find_common_sentences(df, column_name='clean_text', min_length=10, top_n=50):
    all_sentences = []

    for text in df[column_name].dropna():
        sentences = re.split(r'[.!?\n\t]+', str(text))

        for s in sentences:
            cleaned = s.strip()
            cleaned = re.sub(r'\s+', ' ', cleaned)
            if len(cleaned) > min_length:
                all_sentences.append(cleaned)

    common_segments = Counter(all_sentences).most_common(top_n)

    print(f"--- TOP {top_n} COMMON SENTENCE SEGMENTS ---")
    for segment, count in common_segments:
        print(f"Frequency: {count} | Text: {segment}")


def find_common_phrases(df, column_name='final_text', ngram_range=(4, 12), min_df=0.05, top_n=None):
    vectorizer = CountVectorizer(ngram_range=ngram_range, min_df=min_df)
    X = vectorizer.fit_transform(df[column_name].dropna())

    frequencies = zip(vectorizer.get_feature_names_out(), X.toarray().sum(axis=0))
    sorted_frequencies = sorted(frequencies, key=lambda x: x[1], reverse=True)

    if top_n is not None:
        sorted_frequencies = sorted_frequencies[:top_n]

    print(f"--- TOP REPEATED PHRASES (N-grams {ngram_range}) ---")
    for phrase, count in sorted_frequencies:
        print(f"[{count} times]: {phrase}")

### CNN articles cleanup

In [7]:
cnn_real = cnn_real.dropna()
cnn_real = cnn_real.drop_duplicates(subset=['clean_text'])

In [8]:
find_common_sentences(cnn_real, column_name='clean_text', min_length=10, top_n=50)

--- TOP 50 COMMON SENTENCE SEGMENTS ---
Frequency: 222 | Text: cnnindonesia
Frequency: 68 | Text: (mts/bmw/bmw)
Frequency: 67 | Text: mts/bmw/bmw)
Frequency: 51 | Text: dhf/bmw/bmw)
Frequency: 50 | Text: (dhf/bmw/bmw)
Frequency: 49 | Text: (thr/bmw/bmw)
Frequency: 47 | Text: pantauan cnnindonesia
Frequency: 46 | Text: berdasarkan pantauan cnnindonesia
Frequency: 43 | Text: rzr/bmw/bmw)
Frequency: 39 | Text: (rzr/bmw/bmw)
Frequency: 38 | Text: thr/bmw/bmw)
Frequency: 36 | Text: (dmi/bmw/bmw)
Frequency: 35 | Text: cfd/bmw/bmw)
Frequency: 34 | Text: (cfd/bmw/bmw)
Frequency: 30 | Text: dmi/bmw/bmw)
Frequency: 26 | Text: sumber cnnindonesia
Frequency: 22 | Text: mohon maaf, pak prabowo
Frequency: 22 | Text: baca selengkapnya di sini
Frequency: 21 | Text: "dua kali di pilpres juga menang
Frequency: 21 | Text: antara/bmw/bmw)
Frequency: 20 | Text: 200 responden
Frequency: 19 | Text: "dari penampilan kelihatan, banyak kerutan karena mikirin rakyat, ada yang rambutnya putih semua, ada itu
Frequ

In [9]:
def clean_cnn_articles(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'\(?[a-z]{3,7}/[a-z]{3}/[a-z]{3}\)?', '', text)
    text = re.sub(r'\(?[a-z]{3,7}/[a-z]{3}\)?', '', text)
    text = re.sub(r'baca\s*(berita)?\s*(se)?lengkapnya\s*di\s*sini', '', text)
    text = re.sub(r'baca\s*juga\W*', '', text)
    text = re.sub(r'[a-z\s]*cnnindonesia', '', text)
    text = re.sub(r'com,.*?\(\d*/\d*\)', '', text)
    text = re.sub(r'\(cnn\s*[a-z\s]+\)', '', text)

    text = re.sub(r'\s+', ' ', text)  # Normalize spaces
    text = re.sub(r' ,', ',', text)   # Fix loose commas
    text = re.sub(r' \.', '.', text)   # Fix loose periods

    return text.strip()

cnn_real['final_text'] = cnn_real['clean_text'].apply(clean_cnn_articles)

In [10]:
find_common_phrases(cnn_real, column_name='final_text', ngram_range=(4, 12), min_df=0.05)

--- TOP REPEATED PHRASES (N-grams (4, 12)) ---
[778 times]: gubernur dki jakarta anies
[772 times]: dki jakarta anies baswedan
[767 times]: gubernur dki jakarta anies baswedan
[748 times]: presiden joko widodo jokowi
[595 times]: di kompleks parlemen senayan
[581 times]: jawa tengah ganjar pranowo
[577 times]: gubernur jawa tengah ganjar
[575 times]: gubernur jawa tengah ganjar pranowo
[555 times]: kompleks parlemen senayan jakarta
[534 times]: di kompleks parlemen senayan jakarta
[532 times]: komisi pemilihan umum kpu


### Kompas articles cleanup

In [11]:
kompas_real = kompas_real.dropna()
kompas_real = kompas_real.drop_duplicates(subset=['clean_text'])

In [12]:
find_common_sentences(kompas_real, column_name='clean_text', min_length=10, top_n=50)

--- TOP 50 COMMON SENTENCE SEGMENTS ---
Frequency: 16 | Text: aku ya kaget
Frequency: 12 | Text: sementara itu, sandiaga membantah dirinya memberikan sejumlah dana kepada dua parpol pendukungnya
Frequency: 11 | Text: andi mengaku diperintah partainya untuk bicara mengenai dugaan mahar tersebut
Frequency: 11 | Text: bahkan, menurut dia, keputusan demokrat untuk mengungkap soal dugaan mahar ini diambil dalam rapat resmi partai di kediaman ketua umum partai demokrat susilo bambang yudhoyono, rabu (8/8/2018) malam
Frequency: 11 | Text: andi mengaku tidak takut jika pernyataannya di twitter berujung pada konsekuensi hukum
Frequency: 10 | Text: 200 responden
Frequency: 8 | Text: mantan politisi partai solidaritas indonesia (psi) rian ernest bergabung dengan partai golkar menjadi kepala biro pemuda dpd partai golkar dki jakarta
Frequency: 8 | Text: pendaftaran dilakukan satu pintu oleh pengurus pusat partai politik di kpu ri
Frequency: 8 | Text: "kalau misalkan mau hadir tanggal 1 (agustus), 

In [13]:
def clean_kompas_articles(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'(dikutip|berdasarkan)\s*(dari|pantauan)\s*kompas', '', text)
    text = re.sub(r'baca\s*juga\W*', '', text)
    text = re.sub(r'com, selasa', '', text)

    text = re.sub(r'\s+', ' ', text)  # Normalize spaces
    text = re.sub(r' ,', ',', text)   # Fix loose commas
    text = re.sub(r' \.', '.', text)   # Fix loose periods

    return text.strip()

kompas_real['final_text'] = kompas_real['clean_text'].apply(clean_kompas_articles)

In [14]:
find_common_phrases(kompas_real, column_name='final_text', ngram_range=(4, 12), min_df=0.01)

--- TOP REPEATED PHRASES (N-grams (4, 12)) ---
[214 times]: komisi pemilihan umum kpu
[206 times]: presiden dan wakil presiden
[184 times]: komisi pemberantasan korupsi kpk
[182 times]: ketua umum partai gerindra
[173 times]: kompleks parlemen senayan jakarta
[172 times]: partai amanat nasional pan
[167 times]: badan pengawas pemilu bawaslu
[164 times]: di kompleks parlemen senayan
[160 times]: di kompleks parlemen senayan jakarta
[157 times]: terjun ke dunia politik
[138 times]: partai keadilan sejahtera pks
[128 times]: partai gerindra prabowo subianto
[125 times]: ketua umum partai gerindra prabowo
[125 times]: umum partai gerindra prabowo
[123 times]: ketua umum partai gerindra prabowo subianto
[123 times]: umum partai gerindra prabowo subianto
[122 times]: ketua umum partai demokrat
[122 times]: partai kebangkitan bangsa pkb
[116 times]: presiden joko widodo jokowi
[112 times]: ketua umum partai golkar
[111 times]: gubernur dki jakarta anies
[110 times]: partai persatuan pembangun

### Tempo articles cleanup

In [15]:
tempo_real = tempo_real.dropna()
tempo_real = tempo_real.drop_duplicates(subset=['clean_text'])

In [16]:
find_common_sentences(tempo_real, column_name='clean_text', min_length=10, top_n=50)

--- TOP 50 COMMON SENTENCE SEGMENTS ---
Frequency: 351 | Text: ikuti berita terkini dari tempo di google news, klik di sini
Frequency: 319 | Text: m julnis firmansyah
Frequency: 271 | Text: dewi nurita
Frequency: 244 | Text: 12 selanjutnya
Frequency: 95 | Text: janji menangkan anies baswedan dan partai, pks dki: bismillah, allahu akbar, merdeka
Frequency: 79 | Text: co di google news, klik di sini
Frequency: 53 | Text: hendrik khoirul muhid
Frequency: 49 | Text: kronologi 21 hari penyanderaan pilot susi air, mengapa panglima tni akui sulit bebaskan
Frequency: 34 | Text: 123 selanjutnya
Frequency: 30 | Text: julnis firmansyah
Frequency: 28 | Text: septhia ryanthie
Frequency: 27 | Text: dewi nuritaikuti berita terkini dari tempo di google news, klik di sini
Frequency: 25 | Text: ima dini shafira | m
Frequency: 23 | Text: mutia yuantisya
Frequency: 21 | Text: pribadi wicaksono
Frequency: 20 | Text: dukung ganjar jadi capres, pan: baru tahap mendorong, belum sampai tarik kader
Frequency: 2

In [17]:
journalist_names = [
    "m julnis firmansyah", "julnis firmansyah", "sapri maulana",
    "hendrik khoirul muhid", "septhia ryanthie", "mutia yuantisya",
    "pribadi wicaksono", "alfitria nefi pratiwi", "gadis oktaviani",
    "muh raihan muzakki", "risma damayanti", "rahma dwi safitri",
    "naufal ridhwan aly", "annisa firdausi", "fathur rachman",
    "arrijal rachman", "ima dini shafira", "fajar pebrianto",
    "faiz zaki", "delfi ana harahap", "nugroho catur pamungkas",
    "dewi nurita", "mirza bagaskara"
]
journalist_names.sort(key=len, reverse=True)
escaped_names = [re.escape(name) for name in journalist_names]
journalist_regex = re.compile(r'\b(' + '|'.join(escaped_names) + r')\b', flags=re.IGNORECASE)

def clean_tempo_articles(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = journalist_regex.sub("", text)
    text = re.sub(r'(nurita|firmansyah|yuantisya)?ikuti(.*)?di\s*sini', '', text)
    text = re.sub(r'co(.*)?di\s*sini', '', text)
    text = re.sub(r'12(3)?\s*selanjutnya', '', text)
    text = re.sub(r'baca\s*juga\W*', '', text)

    text = re.sub(r'\s+', ' ', text)  # Normalize spaces
    text = re.sub(r' ,', ',', text)   # Fix loose commas
    text = re.sub(r' \.', '.', text)   # Fix loose periods

    return text.strip()

tempo_real['clean_text'] = tempo_real['clean_text'].apply(clean_tempo_articles)

In [18]:
find_common_sentences(tempo_real, column_name='clean_text', min_length=10, top_n=50)

--- TOP 50 COMMON SENTENCE SEGMENTS ---
Frequency: 93 | Text: janji menangkan anies baswedan dan partai, pks dki: bismillah, allahu akbar, merdeka
Frequency: 49 | Text: kronologi 21 hari penyanderaan pilot susi air, mengapa panglima tni akui sulit bebaskan
Frequency: 34 | Text: jokowi hadiri groundbreaking plta mentarang induk kalimantan utara hari ini
Frequency: 27 | Text: dukung ganjar jadi capres, pan: baru tahap mendorong, belum sampai tarik kader
Frequency: 20 | Text: 500 per liter
Frequency: 15 | Text: 150 per liter menjadi rp6
Frequency: 15 | Text: 500 menjadi rp14
Frequency: 13 | Text: arahan jokowi: ekosistem kendaraan listrik indonesia bisa saingi thailand
Frequency: 13 | Text: mario dandy dijerat pasal 351, mahfud minta terapkan pasal 354 dan 355, apa bedanya
Frequency: 12 | Text: metodologi yang digunakan adalah metode acak bertingkat (multistage random sampling) pada tingkat kepercayaan 95 persen
Frequency: 12 | Text: ,” kata dia
Frequency: 12 | Text: 650 per liter menjadi

In [19]:
find_common_phrases(tempo_real, column_name='clean_text', ngram_range=(4, 12), min_df=0.05)

--- TOP REPEATED PHRASES (N-grams (4, 12)) ---
[401 times]: groundbreaking plta mentarang induk
[401 times]: groundbreaking plta mentarang induk kalimantan
[401 times]: groundbreaking plta mentarang induk kalimantan utara
[401 times]: groundbreaking plta mentarang induk kalimantan utara hari
[401 times]: groundbreaking plta mentarang induk kalimantan utara hari ini
[401 times]: hadiri groundbreaking plta mentarang
[401 times]: hadiri groundbreaking plta mentarang induk
[401 times]: hadiri groundbreaking plta mentarang induk kalimantan
[401 times]: hadiri groundbreaking plta mentarang induk kalimantan utara
[401 times]: hadiri groundbreaking plta mentarang induk kalimantan utara hari
[401 times]: hadiri groundbreaking plta mentarang induk kalimantan utara hari ini
[401 times]: induk kalimantan utara hari
[401 times]: induk kalimantan utara hari ini
[401 times]: jokowi hadiri groundbreaking plta
[401 times]: jokowi hadiri groundbreaking plta mentarang
[401 times]: jokowi hadiri groundbre

In [20]:
def clean_tempo_articles(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    pattern = r'(ikuti)?\W*berita\W*terkini\W*dari\W*tempo\W*[A-Za-z0-9\s]+\b'
    text = re.sub(pattern, '', text)
    pattern = r'\bikuti\W*klik\W*di\W*sini'
    text = re.sub(pattern, '', text)

    text = re.sub(r'\s+', ' ', text)  # Normalize spaces
    text = re.sub(r' ,', ',', text)   # Fix loose commas
    text = re.sub(r' \.', '.', text)   # Fix loose periods

    return text.strip()

tempo_real['final_text'] = tempo_real['clean_text'].apply(clean_tempo_articles)

### Hoax articles cleanup

In [21]:
tbh_fake = tbh_fake.dropna()
tbh_fake = tbh_fake.drop_duplicates(subset=['clean_text'])

In [22]:
find_common_sentences(tbh_fake, column_name='clean_text', min_length=10, top_n=50)

--- TOP 50 COMMON SENTENCE SEGMENTS ---
Frequency: 252 | Text: ” = = = = =
Frequency: 79 | Text: selengkapnya di bagian
Frequency: 71 | Text: selengkapnya terdapat di
Frequency: 26 | Text: terima kasih
Frequency: 17 | Text: assalamualaikum
Frequency: 13 | Text: terimakasih
Frequency: 10 | Text: breaking news
Frequency: 10 | Text: […] (narasi dilanjutkan setelah bagian referensi) = = = = =
Frequency: 9 | Text: […] (lanjutan narasi setelah bagian referensi) = = = = =
Frequency: 9 | Text: semoga bermanfaat
Frequency: 9 | Text: selengkapnya di bagian pembahasan
Frequency: 8 | Text: melalui kuesioner, anda akan memiliki kesempatan untuk mendapatkan 2000000 rupiah
Frequency: 8 | Text: ”[…](narasi dilanjutkan setelah bagian referensi)
Frequency: 8 | Text: = eksekutif
Frequency: 8 | Text: = (all) berarti pulang & pergi
Frequency: 7 | Text: alhamdulillah
Frequency: 7 | Text: selengkapnya ada di
Frequency: 6 | Text: assalamualaikum wr wb
Frequency: 6 | Text: tersebut tidak benar
Frequency: 6 | T

In [23]:
def clean_tbh_articles(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'selengkapnya\s*(terdapat|ada)?\s*di\W*', '', text)
    text = re.sub(r'tersebut\s*tidak\s*benar\W*', '', text)
    text = re.sub(r'selengkapnya\s*pada\W*', '', text)
    text = re.sub(r'\W*(lanjutan)?\W*narasi\W*', '', text)
    text = re.sub(r'=', '', text)

    text = re.sub(r'\s+', ' ', text)  # Normalize spaces
    text = re.sub(r' ,', ',', text)   # Fix loose commas
    text = re.sub(r' \.', '.', text)   # Fix loose periods

    return text.strip()

tbh_fake['final_text'] = tbh_fake['clean_text'].apply(clean_tbh_articles)

In [24]:
find_common_sentences(tbh_fake, column_name='final_text', min_length=10, top_n=50)

--- TOP 50 COMMON SENTENCE SEGMENTS ---
Frequency: 26 | Text: terima kasih
Frequency: 17 | Text: assalamualaikum
Frequency: 14 | Text: terimakasih
Frequency: 10 | Text: breaking news
Frequency: 9 | Text: semoga bermanfaat
Frequency: 9 | Text: bagian pembahasan
Frequency: 8 | Text: sumber: twitter arsip
Frequency: 8 | Text: melalui kuesioner, anda akan memiliki kesempatan untuk mendapatkan 2000000 rupiah
Frequency: 8 | Text: (all) berarti pulang & pergi
Frequency: 7 | Text: alhamdulillah
Frequency: 7 | Text: sumber: whatsapp arsip
Frequency: 6 | Text: assalamualaikum wr wb
Frequency: 6 | Text: salam sehat
Frequency: 6 | Text: subhanallah
Frequency: 6 | Text: eksekutif & bisnis
Frequency: 5 | Text: sumber: facebook arsip
Frequency: 5 | Text: mengejutkan
Frequency: 5 | Text: karena di negara itu ada kira2 200 orang pengidap aids kerja di pabrik kalengan, dan mereka masukkan darah mereka ke dalam kalengan2 itu, dan saat ini masalah tersebut telah diketahui depkes thailand sehingga kaleng2a

In [25]:
find_common_phrases(tbh_fake, column_name='final_text', ngram_range=(4, 12), min_df=0.005)

--- TOP REPEATED PHRASES (N-grams (4, 12)) ---
[146 times]: ke dalam bahasa indonesia
[142 times]: diterjemahkan ke dalam bahasa
[142 times]: diterjemahkan ke dalam bahasa indonesia
[49 times]: diterjemahkan ke bahasa indonesia


In [26]:
def clean_tbh_articles(text):
    if not isinstance(text, str):
        return text

    text = re.sub(r'(diterjemahkan)?\W*ke\W*(dalam)?\W*bahasa\W*', '', text)

    pattern = r'(gambar|foto|hoax|sumber|akun|facebook|akun\s*facebook|tidak\s*benar|terjemahan|video|judul|faktanya)'
    text = re.sub(pattern, '', text)
    pattern = r'(berita|beredar|postingan|unggah|the|yang\s*salah|ternyata|bagian|palsu|artikel)'
    text = re.sub(pattern, '', text)

    text = re.sub(r'\s+', ' ', text)  # Normalize spaces
    text = re.sub(r' ,', ',', text)   # Fix loose commas
    text = re.sub(r' \.', '.', text)   # Fix loose periods

    return text.strip()

tbh_fake['final_text'] = tbh_fake['final_text'].apply(clean_tbh_articles)

## Dataset Preparation

All data that has undergone the cleaning process is combined into a single master dataset. The distribution of target classes (`labels`) is checked to identify any potential data imbalance that could distort the loss function during model training.

In [27]:
dataset = pd.concat([cnn_real, kompas_real, tempo_real, tbh_fake]).sample(frac = 1, random_state=42)

In [28]:
dataset = dataset.dropna(subset=['final_text'])
dataset = dataset.drop_duplicates(subset='final_text')

In [29]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26894 entries, 3913 to 3765
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  26894 non-null  int64  
 1   Title       26894 non-null  object 
 2   Timestamp   26894 non-null  object 
 3   FullText    26894 non-null  object 
 4   Tags        26894 non-null  object 
 5   Author      26894 non-null  object 
 6   Url         26894 non-null  object 
 7   text_new    26894 non-null  object 
 8   hoax        26894 non-null  int64  
 9   clean_text  26894 non-null  object 
 10  final_text  26894 non-null  object 
 11  politik     6390 non-null   float64
 12  Narasi      6390 non-null   object 
dtypes: float64(1), int64(2), object(10)
memory usage: 2.9+ MB


In [30]:
dataset['hoax'].value_counts()

,count
hoax,
0,20504
1,6390


In [31]:
dataset = dataset.rename(columns={"hoax": "labels"})
X = dataset['final_text']
y = dataset['labels']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)} | Validation samples: {len(X_val)}")

Training samples: 21515 | Validation samples: 5379


## Baseline Model (TF-IDF + Logistic Regression)
Before moving on to deep learning architectures, a *baseline* model was built using a combination of **TF-IDF Vectorizer** and **Logistic Regression**.

To find the best configuration, **5-Fold Stratified Cross-Validation (`GridSearchCV`)** was applied to test 48 candidate hyperparameter combinations, covering the *n-gram* range, document frequency thresholds (`min_df` and `max_df`), the regularization constant (`C`), and the balancing class weight.

In [32]:
baseline_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])

param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [3, 5],
    'tfidf__max_df': [0.7, 0.85],

    'lr__C': [0.1, 1.0, 10.0],
    'lr__penalty': ['l2'],
    'lr__class_weight': ['balanced', None]
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=baseline_pipeline,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)
print(f"Best Macro F1-Score: {grid_search.best_score_:.4f}")
print("Best Parameters:", grid_search.best_params_)

Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best Macro F1-Score: 0.9754
Best Parameters: {'lr__C': 10.0, 'lr__class_weight': None, 'lr__penalty': 'l2', 'tfidf__max_df': 0.85, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 2)}


In [33]:
y_pred = grid_search.predict(X_val)
roc_auc = roc_auc_score(y_val, y_pred)

print(f"Baseline Accuracy: {accuracy_score(y_val, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc}\n")
print(classification_report(y_val, y_pred, target_names=['Real News', 'Fake News']))

Baseline Accuracy: 0.9822
ROC-AUC: 0.9732139075205521

              precision    recall  f1-score   support

   Real News       0.99      0.99      0.99      4101
   Fake News       0.97      0.96      0.96      1278

    accuracy                           0.98      5379
   macro avg       0.98      0.97      0.98      5379
weighted avg       0.98      0.98      0.98      5379



### Baseline Model Evaluation

The best coefficient values from the baseline model are extracted to identify which words or text fragments have the highest correlation in driving the model's decision toward the **FAKE NEWS (Class 1)** classification. This step is crucial to ensure the model captures the linguistic characteristics of hoaxes (such as informal language, provocative terms, or conspiracy references).

In [34]:
baseline_pipeline = grid_search.best_estimator_
tfidf_step = baseline_pipeline.named_steps['tfidf']
lr_step = baseline_pipeline.named_steps['lr']

feature_names = np.array(tfidf_step.get_feature_names_out())

print(lr_step.classes_)
coefficients = lr_step.coef_[0]

df_coefficients = pd.DataFrame({
    'Word/Phrase': feature_names,
    'Coefficient': coefficients
})

top_fake_words = df_coefficients.sort_values(by='Coefficient', ascending=False).head(20)
print("--- Top 20 Words Predicting FAKE NEWS ---")
print(top_fake_words.to_string(index=False))

[0 1]
--- Top 20 Words Predicting FAKE NEWS ---
Word/Phrase  Coefficient
         yg    11.824513
       anda     5.375648
        ber     4.705586
        nya     4.629675
    yang di     4.406559
        gak     4.144090
        pki     3.781411
    tanggal     3.778034
     kalian     3.707760
         dr     3.517200
       2020     3.491792
      virus     3.457764
        dgn     3.388827
         in     3.383496
       info     3.382789
     wanita     3.267266
     muslim     3.266341
       bank     3.256895
  thumbnail     3.247065
     semoga     3.222512


In [35]:
sample_hoax = "Uang kertas pecahan Rp100.000 tidak berlaku lagi mulai besok pagi karena inflasi drastis."

prediction = baseline_pipeline.predict([sample_hoax])
probabilities = baseline_pipeline.predict_proba([sample_hoax])

print(f"Prediction: {prediction[0]} (0=Real, 1=Fake)")
print(f"Confidence: {probabilities[0]}")

Prediction: 1 (0=Real, 1=Fake)
Confidence: [0.07695314 0.92304686]


In [36]:
joblib.dump(baseline_pipeline, "baseline.pkl")
size_mb = os.path.getsize("baseline.pkl") / (1024 * 1024)
print(size_mb)

4.284183502197266


In [37]:
sample = X_val.iloc[0:1]
start = time.perf_counter()
baseline_pipeline.predict(sample)
end = time.perf_counter()

print(f"{(end - start) * 1000:.2f} ms")

3.33 ms


## Deep Learning Model Architecture (Multi-Scale Conv1D)
To capture sequential local structures and phrasal context within news paragraphs, a neural network architecture based on a **1D Convolutional Neural Network (Conv1D)** was developed.

### Experiment Workflow:
1. **Text Length Analysis:** Analyzing word length quantiles to determine the optimal `MAX_SEQUENCE_LENGTH = 512` (covering 98%+ of the data without extreme truncation).
2. **Class Imbalance Scaling:** Mathematically calculating class weights using the *inverse frequency* formula to impose a higher loss penalty on the minority class (Hoaxes).
3. **Feature Extraction Architecture:** Utilizing three consecutive `Conv1D` layers with varying kernel sizes (3, 5, and 7) to capture text patterns at multiple scales, followed by `GlobalMaxPool1D` to reduce the temporal dimension.

In [38]:
lengths = X_train.str.split().apply(len)
print(lengths.describe())

count    21515.000000
mean       179.757797
std        149.687908
min          1.000000
25%         46.000000
50%        178.000000
75%        264.000000
max       3478.000000
Name: final_text, dtype: float64


In [39]:
print(lengths.quantile([0.5, 0.9, 0.95, 0.99]))

0.50    178.00
0.90    348.00
0.95    415.00
0.99    628.86
Name: final_text, dtype: float64


In [40]:
MAX_VOCAB_SIZE = 10000
MAX_SEQUENCE_LENGTH = 512
OOV_TOK = "<OOV>"

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token=OOV_TOK)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQUENCE_LENGTH, padding="post")

X_val_seq = tokenizer.texts_to_sequences(X_val)
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_SEQUENCE_LENGTH, padding="post")

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = dict(enumerate(class_weights_array))
print("Calculated Class Weights:", class_weight_dict)

Calculated Class Weights: {0: np.float64(0.6558251539352558), 1: np.float64(2.104362284820031)}


In [41]:
EMBEDDING_DIM = 128

model = Sequential([
    tf.keras.Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32),
    Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=EMBEDDING_DIM),
    Conv1D(128, 3, activation='relu', padding='same'),
    Conv1D(128, 5, activation='relu', padding='same'),
    Conv1D(64, 7, activation='relu', padding='same'),
    GlobalMaxPool1D(),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=[
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc',
    mode='max',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=20,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=[early_stop]
)

Epoch 1/20
337/337 ━━━━━━━━━━━━━━━━━━━━ 20s 39ms/step - auc: 0.9873 - loss: 0.1423 - precision: 0.8701 - recall: 0.9356 - val_auc: 0.9952 - val_loss: 0.0985 - val_precision: 0.8958 - val_recall: 0.9890
Epoch 2/20
337/337 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - auc: 0.9986 - loss: 0.0357 - precision: 0.9689 - recall: 0.9881 - val_auc: 0.9942 - val_loss: 0.1091 - val_precision: 0.8905 - val_recall: 0.9859
Epoch 3/20
337/337 ━━━━━━━━━━━━━━━━━━━━ 9s 27ms/step - auc: 0.9998 - loss: 0.0101 - precision: 0.9882 - recall: 0.9975 - val_auc: 0.9872 - val_loss: 0.2040 - val_precision: 0.8789 - val_recall: 0.9828
Epoch 4/20
337/337 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - auc: 0.9996 - loss: 0.0125 - precision: 0.9864 - recall: 0.9957 - val_auc: 0.9855 - val_loss: 0.3127 - val_precision: 0.7940 - val_recall: 0.9922
Epoch 5/20
337/337 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - auc: 0.9999 - loss: 0.0035 - precision: 0.9963 - recall: 0.9996 - val_auc: 0.9855 - val_loss: 0.1276 - val_precision: 0.9707 - val_recall: 

## Threshold Tuning & Post-Training Quantization
Model performance is not measured solely by raw accuracy, but by the optimal classification threshold that minimizes misclassification.

* **Threshold Tuning:** Iterating the probability threshold values from 0.1 to 0.9 on the validation data to find the cutoff point that yields the **highest Macro F1-Score**.
* **Model Quantization:** Converting the base Keras model (`.keras`) to the TensorFlow Lite format (`.tflite`) using default quantization optimizations to drastically reduce binary size and accelerate *inference latency*, making it ideal for running on Android mobile applications.

In [42]:
y_prob = model.predict(X_val_pad)
thresholds = np.arange(0.1, 0.9, 0.01)
best_threshold = 0.5
best_f1 = 0

for t in thresholds:
    preds = (y_prob > t).astype(int)
    score = f1_score(y_val, preds)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

print(best_threshold)

169/169 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
0.7599999999999997


In [43]:
model.save("HoaXGY_model.keras")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
with open("HoaXGY_model.tflite", "wb") as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmpmio78e50'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 512), dtype=tf.int32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136245770208656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136245770210768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136245770210192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136245770208848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136245770209040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136245770211728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136245770211920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136245770211536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136245770212688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136245770213072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136245770210960: TensorS

In [44]:
keras_model = tf.keras.models.load_model("HoaXGY_model.keras")
interpreter = tf.lite.Interpreter(model_path="HoaXGY_model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [45]:
keras_probs = keras_model.predict(X_val_pad, verbose=0).flatten()

tflite_probs = []
for sample in X_val_pad:
    input_data = np.expand_dims(
        sample.astype(np.int32),
        axis=0
    )
    interpreter.set_tensor(
        input_details[0]['index'],
        input_data
    )
    interpreter.invoke()

    output = interpreter.get_tensor(
        output_details[0]['index']
    )

    tflite_probs.append(output[0][0])

tflite_probs = np.array(tflite_probs)

diff = np.abs(keras_probs - tflite_probs)
print(f"Mean Difference: {diff.mean():.8f}")
print(f"Max Difference: {diff.max():.8f}")

Mean Difference: 0.00019327
Max Difference: 0.00715214


In [46]:
keras_pred = (keras_probs >= best_threshold).astype(int)
tflite_pred = (tflite_probs >= best_threshold).astype(int)

print(f"Keras Accuracy: {accuracy_score(y_val, keras_pred)}")
print(f"Keras ROC-AUC: {roc_auc_score(y_val, keras_probs)}\n")

print(f"TFLite Accuracy: {accuracy_score(y_val, tflite_pred)}")
print(f"TFLite ROC-AUC: {roc_auc_score(y_val, tflite_probs)}")

Keras Accuracy: 0.9801078267335936
Keras ROC-AUC: 0.9967804333383322

TFLite Accuracy: 0.9801078267335936
TFLite ROC-AUC: 0.9967857757507138


In [47]:
print("Keras Confusion Matrix")
print(classification_report(y_val, keras_pred))
print("\nTFLite Confusion Matrix")
print(classification_report(y_val, tflite_pred))

Keras Confusion Matrix
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4101
           1       0.94      0.97      0.96      1278

    accuracy                           0.98      5379
   macro avg       0.97      0.98      0.97      5379
weighted avg       0.98      0.98      0.98      5379


TFLite Confusion Matrix
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4101
           1       0.94      0.97      0.96      1278

    accuracy                           0.98      5379
   macro avg       0.97      0.98      0.97      5379
weighted avg       0.98      0.98      0.98      5379



In [48]:
keras_size = (os.path.getsize("HoaXGY_model.keras") / (1024*1024))
tflite_size = (os.path.getsize("HoaXGY_model.tflite") / (1024*1024))

print(f"Keras size: {keras_size:.2f} MB")
print(f"TFLite size: {tflite_size:.2f} MB")

Keras size: 16.88 MB
TFLite size: 1.41 MB


In [49]:
sample = X_val_pad[0:1].astype(np.int32)

# warmup
for _ in range(10):
    keras_model.predict(sample, verbose=0)

n_runs = 100
start = time.perf_counter()
for _ in range(n_runs):
    keras_model.predict(sample, verbose=0)
end = time.perf_counter()
keras_ms = (end - start) / n_runs * 1000
print(f"Keras: {keras_ms:.2f} ms")

Keras: 83.55 ms


In [50]:
sample = X_val_pad[0:1].astype(np.int32)

# warmup
for _ in range(10):
    interpreter.set_tensor(
        input_details[0]['index'],
        sample
    )
    interpreter.invoke()

n_runs = 100
start = time.perf_counter()
for _ in range(n_runs):
    interpreter.set_tensor(
        input_details[0]['index'],
        sample
    )
    interpreter.invoke()

end = time.perf_counter()
tflite_ms = (end - start) / n_runs * 1000
print(f"TFLite: {tflite_ms:.2f} ms")

TFLite: 2.74 ms
